# Poysuwop_Nr1 / Ainu Tokenizer
Transformer based tokenizer

### 0. Library install

In [165]:
!pip install torch accelerate transformers tokenizers huggingface sentencepiece

  Using cached accelerate-0.25.0-py3-none-any.whl (265 kB)


### 1. Preprocess

In [ ]:
# Library
import os
import glob
import json
import re
import collections
import unicodedata
import pprint

# Change Current Directory
os.chdir('/content/drive/MyDrive/Colab Notebooks/MasterStudy')

#### Load dataset

In [ ]:
def alphabetCheck(text):
    for char in text:
        if not (char.isascii() and (char.isalpha() or char.isspace() or '=' or "'")):
            return False
    return True

In [ ]:
# Fetch sentence using the 'ain' tag from .json file
# ['ain']       romanized Ainu sentence
# ['ain-kana']  Ainu sentence transcribed in Ainu Kana
# ['jpn']       Japanese translation

texts = []

for f in glob.glob('./corpus_json/*.json'):

    # load .json
    data = json.load(open(str(f)))

    for key, value in data.items():

        if key != 'code' and key != 'title':
            sentence = value['ain']

            if len(sentence) > 0 and alphabetCheck(sentence) == True:
                texts.append(sentence)

In [ ]:
##### Check charcters used in the dataset

charlist = [char for text in texts for char in list(text)]
charlist = [k for k, v in collections.Counter(charlist).items() if v > 1]

#charlist = [re.sub(r"[^a-zA-Z0-9 =]","",char) for char in charlist]
charlist_TBR = ''.join(str(x) for x in charlist)
char_TBR = re.sub(r"[a-zA-Z0-9 =]","",charlist_TBR)
print("Charcaters to be removed: {0}".format(char_TBR))

Charcaters to be removed: .'?}!_(),:"-
[]`*;/@


#### Cleansing

In [ ]:
# Affix cleansing

dicPrefixes = {
    'ku=':"ku= ",
    'k=':'k= ',
    'e=':'e= ',
    'eci=':'eci= ',
    'a=':'a= ',
    'c=':'c= ',
    'ci=':'ci= ',
    'i=':'i= ',
    'en=':'en= ',
    'un=':'un= ',
    'an=':"an= ",
}

dicSuffixes = {
    '=as':" =as",
    '=an':" =an",
}

def ainAffixCleanse(sentence, dic_chars, mode):
    # mode: 0 - prefix, 1 - suffix

    if mode == 0:
        for key in dic_chars.keys():
            sentence = re.sub(r'(\s|^)' + re.escape(key), r'\1' +dic_chars[key], sentence)
    else:
        for key in dic_chars.keys():
            sentence = re.sub(re.escape(key) + r'(?=\s|$)', dic_chars[key], sentence)

    # 人称接辞と同音異義形態素で構成される語のうち、an=anだけ特殊処理
    # 稀なケースだが、包括的一人称接辞としてan=が使われると、動詞 an と
    # Both ku=ku (I drink) and e=e (you eat) are not subject to be addressed, as they can be correctly split into the prefix and the verb.
    dic_revise = {
        "an= an": "an =an",
    }

    for key in dic_revise.keys():
        sentence = sentence.replace(key, dic_revise[key]) if key in sentence else sentence

    sentence = ' '.join(sentence.split())

    return sentence

sentence = 'an=an'
sentence = ainAffixCleanse(sentence, dicPrefixes, 0)
sentence = ainAffixCleanse(sentence, dicSuffixes, 1)
sentence = ainAffixCleanse(sentence, dicPrefixes, 0) # 主格接辞と目的接辞を2つ抱合している語があるため、2度目の処理
sentence = ainAffixCleanse(sentence, dicSuffixes, 1) # 主格接辞と目的接辞を2つ抱合している語があるため、2度目の処理

sentence

'an =an'

In [ ]:
# Orthography - a=kor itak
dicOrthography = {
    #'ai':'ay',
    #'au':'aw',
    #'ei':'ey',
    #'eu':'ew',
    #'iu':'iw',
    #'oi':'oy',
    #'ou':'ow',
    #'ui':'uy',
    'ch':'c',
    'sh':'s',
}

def char_cleanse(sentence, dic_chars):

    for key in dic_chars.keys():
        sentence = sentence.replace(key, dic_chars[key]) if key in sentence else sentence

    # Remove Extra Spaces
    sentence = ' '.join(sentence.split())
    return sentence

In [ ]:
def charCleanse(sentence):

    # Zenkaku -> Hankaku equivalent
    sentence = unicodedata.normalize('NFKD', sentence)

    # Remove number with parentheses (eg. (123) --> '', [123] --> '')
    sentence = re.sub(r'\(.*?\)|\[.*?\]', '', sentence)

    # Remove symbols
    sentence = re.sub(r"[^a-zA-Z0-9 =]","",sentence)

    # Apply orthography
    sentence = char_cleanse(sentence, dicOrthography)

    # Clease affixes
    sentence = ainAffixCleanse(sentence, dicPrefixes, 0)
    sentence = ainAffixCleanse(sentence, dicSuffixes, 1)
    sentence = ainAffixCleanse(sentence, dicPrefixes, 0) # 主格接辞と目的接辞を2つ抱合している語があるため、2度目の処理
    sentence = ainAffixCleanse(sentence, dicSuffixes, 1) # 主格接辞と目的接辞を2つ抱合している語があるため、2度目の処理

    # Trim sentence
    sentence = sentence.strip()

    return sentence

# -- ex. --
sentence = 'yayán ainu ku=ne ruwe ne korka, ainu itak ani Transformer[123] ku=kor_ rushuy un!'
charCleanse(sentence)

'yayan ainu ku= ne ruwe ne korka ainu itak ani Transformer ku= kor rusuy un'

In [ ]:
# Cleanse sentences
for sentence in texts:
    # Clease sentence
    sentence = charCleanse(sentence)

In [ ]:
# sample
# sentence = 'ク・ネㇷ゚キ　ヒ　カ　エン・ココパンパ　コㇿカ　ネ　ワ　アンペ　カ　ケシㇼキラㇷ゚　ワ　ク・ネㇷ゚キ　ルスイ　ワ　ケシㇼキラㇷ゚　ワ　クス　（ク・ネㇷ゚キ）　チセ　ソイ　ペカ　クネㇷ゚キ　コㇿ　カン。'
# sentence = 'ペウレクﾙ　ネ　コﾛカ　ア・アイヌコﾛ'
sentence = '“Shirokanipe ranran pishkan, konkanipe ranran pishkan.” arian rekpo chiki kane petesoro sapash aine, ainukotan enkashike chikush kor shichorpokun inkarash ko teeta wenkur tane nishpa ne, teeta nishpa tane wenkur ne kotom shiran.'
sentence_orth = Cleanse(sentence)

print(sentence)
print(sentence_orth)

“Shirokanipe ranran pishkan, konkanipe ranran pishkan.” arian rekpo chiki kane petesoro sapash aine, ainukotan enkashike chikush kor shichorpokun inkarash ko teeta wenkur tane nishpa ne, teeta nishpa tane wenkur ne kotom shiran.
Shirokanipe ranran piskan konkanipe ranran piskan arian rekpo ciki kane petesoro sapas aine ainukotan enkasike cikus kor sicorpokun inkaras ko teeta wenkur tane nispa ne teeta nispa tane wenkur ne kotom siran


#### Affix Marker Check

In [ ]:
# 人称接辞マーカー = の両端に文字が残っているものの抽出
for i in range(len(texts)):
    if re.search(r'\S+=\S+', texts[i]):
        print(texts[i])

sekor yaynu=anpa hike ka
uturasinot=anpa ne ya arukesanpa =an
na nen nen iki=anpa kor oka =an hike ka
ipe=anpa hine ipe oka an rapok
uteksam cipkuste=anpa wa
hotke=anpa hike ka
sinna sinna ka ekimne=anpa ka ki
su kim ta suke=anpa wa
pohene ucistaspare=anpa kor oka =an a korka
ipe=anpa kor oka =an rapok
ora ukoomanne sakekar=anpa wa
po hene kimoterke=a
inaw oa=asi ne ya ki kor an =an ani
earkinne arwen sakanramu aci=kore wa
pon-a=yupi poro a= yupi a= saha sekor hawean =an kor
aci=tekekar wa
paye=anrewsi =an pa ayne
oraun inawke=aninawroski =an kusu inawke =an kor
aci=nure hawe
ora hopuni=a hopuni =an hine
mourii=kore wa manpuri ne a= kor
onne utar tak wa asa=suke wa suke =an wa a= ipere
ari an pe aya=ye konno
ani=hotuyekar konno ahup =an wa a= tononte
an-i=tura hine soy peka kim peka
sirkunpato asihikea=sike an= esiyarpokani wa
arpa =an rusuy neuna=eyaynita yakka
sekor yaynu=anpa hike ka
uturasinot=anpa ne ya arukesanpa =an
na nen nen iki=anpa kor oka =an hike ka
ipe=anpa hine ipe oka a

#### Save cleansed sentences

In [ ]:
# Temporary saving to a txt file
with open("./ain.txt", "w") as output:
    output.write(str(texts))

print(len(texts))

144866


### 2. Tokenizer準備

In [ ]:
from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer
)

# Initialize a tokenizer
#tokenizer = ByteLevelBPETokenizer()
tokenizer = Tokenizer(models.WordPiece(unk_token='[UNK]'))

tokenizer.normalizer = normalizers.Sequence(
    [normalizers.Lowercase(), normalizers.NFKC()]
)

tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [ ]:
def yield_text():
    for row in texts:
        yield row

In [ ]:
trainer = trainers.WordPieceTrainer(
    vocab_size=30_000,
    special_tokens=['[UNK]', '[PAD]', '[CLS]', '[SEP]', '[MASK]'],
    min_frequency=2,
    continuing_subword_prefix='##'
)

In [ ]:
tokenizer.train_from_iterator(yield_text(), trainer=trainer)

In [ ]:
# Vocabulary size
tokenizer.get_vocab_size()

17420

In [ ]:
cls_id = tokenizer.token_to_id('[CLS]')
sep_id = tokenizer.token_to_id('[SEP]')

In [ ]:
tokenizer.post_processor = processors.TemplateProcessing(
    single=f'[CLS]:0 $A:0 [SEP]:0',
    pair=f'[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1',
    special_tokens=[
        ('[CLS]', cls_id),
        ('[SEP]', sep_id)
    ]
)

In [ ]:
tokenizer.decoder = decoders.WordPiece(prefix='##')

In [ ]:
from transformers import PreTrainedTokenizerFast

full_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token='[UNK]',
    pad_token='[PAD]',
    cls_token='[CLS]',
    sep_token='[SEP]',
    mask_token='[MASK]'
)

#### Save tokenizer model

In [ ]:
full_tokenizer.save_pretrained('ain-base-tokenizer')

('ain-base-tokenizer/tokenizer_config.json',
 'ain-base-tokenizer/special_tokens_map.json',
 'ain-base-tokenizer/tokenizer.json')

#### Load Ainu

In [ ]:
tokenizer = PreTrainedTokenizerFast.from_pretrained('ain-base-tokenizer')

In [ ]:
# Encoding - Nr1
sentence = "ohonno somo unukar=an"
print('Tokenizer results: ', tokenizer(sentence))

encodedTokens = tokenizer.encode(sentence)
print('Encoding: ', encodedTokens)

decodedTokens = tokenizer.decode(encodedTokens)
print('Decoding: ', decodedTokens)

Tokenizer results:  {'input_ids': [2, 1545, 206, 2730, 28, 108, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}
Encoding:  [2, 1545, 206, 2730, 28, 108, 3]
Decoding:  [CLS] ohonno somo unukar = an [SEP]


In [ ]:
# Encoding - Nr2
sentence = "Ku=kor wa k=an an pe hinak ta k=anu yakka k=erampewtek"
print('Tokenizer results: ', tokenizer(sentence))

encodedTokens = tokenizer.encode(sentence)
print('Encoding: ', encodedTokens)

decodedTokens = tokenizer.decode(encodedTokens)
print('Decoding: ', decodedTokens)

Tokenizer results:  {'input_ids': [2, 227, 28, 116, 115, 45, 28, 108, 108, 136, 811, 120, 45, 28, 735, 205, 45, 28, 858, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Encoding:  [2, 227, 28, 116, 115, 45, 28, 108, 108, 136, 811, 120, 45, 28, 735, 205, 45, 28, 858, 3]
Decoding:  [CLS] ku = kor wa k = an an pe hinak ta k = anu yakka k = erampewtek [SEP]
